[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-tsne.ipynb)

# t-SNE & UMAP — Looking at High-Dimensional Data

*AIBits Academy · Machine Learning End To End · Dimensionality Reduction*

PCA gives you a straight-line view of your data. When the structure is curved, t-SNE and UMAP give you a map — and a map can mislead you if you read it as a photograph.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

In [ ]:
%pip install -q umap-learn

> **📋 Real-World Case Study — 1,797 Handwritten Digits**
>
> The scikit-learn digits dataset holds 1,797 8×8 images of the digits 0–9, so each image is a point in 64 dimensions. Projected onto the two strongest PCA directions, the ten digits overlap in a smear: a 10-nearest-neighbour classifier working only from those two coordinates gets **52.0%** of digits right. The same 64 numbers laid out by t-SNE recover **95.2%** — higher than the 93.7% obtained from all 64 original features, plausibly because the map smooths away pixel-level noise in the neighbourhoods. The rest of this page explains why, how to tune the method, and the five ways such maps are commonly misread.

## Why PCA Is Not Always Enough

PCA (the previous pages) finds the straight directions of greatest variance. If the classes sit on curved or twisted surfaces in high-dimensional space — a rolled-up sheet, a ring, one cluster wrapped around another — no straight projection can unroll them, and Kernel PCA needs a kernel you must choose. **t-SNE** (t-distributed Stochastic Neighbour Embedding, van der Maaten & Hinton, 2008) and **UMAP** (McInnes et al., 2018) take a different approach: they do not try to preserve global geometry at all. They try to preserve *who is near whom*, and lay the points out in 2-D so that each point’s neighbours stay its neighbours.

## How t-SNE Works

**Step 1 — neighbour probabilities in the original space.** For each point *i*, t-SNE places a Gaussian around it and turns distances into the probability that *i* would pick *j* as a neighbour. The Gaussian’s width is chosen per point so that each point has the same *perplexity*, roughly “the effective number of neighbours” it pays attention to.

$$p_{j\mid i} = \dfrac{\exp\!\left(-\lVert \mathbf{x}_i-\mathbf{x}_j\rVert^2 / 2\sigma_i^2\right)}{\sum_{k\ne i}\exp\!\left(-\lVert \mathbf{x}_i-\mathbf{x}_k\rVert^2 / 2\sigma_i^2\right)} \qquad p_{ij}=\dfrac{p_{j\mid i}+p_{i\mid j}}{2n}$$

**Step 2 — neighbour probabilities on the 2-D map.** The map uses a heavy-tailed Student-t distribution (one degree of freedom), which lets moderately distant points sit far apart without penalty and is the reason clusters separate cleanly:

$$q_{ij} = \dfrac{\left(1+\lVert \mathbf{y}_i-\mathbf{y}_j\rVert^2\right)^{-1}}{\sum_{k\ne l}\left(1+\lVert \mathbf{y}_k-\mathbf{y}_l\rVert^2\right)^{-1}}$$

**Step 3 — move the points to make the two agree.** Gradient descent moves the 2-D points to minimise the Kullback–Leibler divergence between the two sets of probabilities:

$$C = \operatorname{KL}(P\,\Vert\,Q) = \sum_{i\ne j} p_{ij}\,\ln\dfrac{p_{ij}}{q_{ij}}$$

Because the cost punishes putting true neighbours far apart much more than putting non-neighbours close together, t-SNE keeps local neighbourhoods intact and is relaxed about everything else — which is exactly why its *distances between clusters* should not be trusted (see below).

## PCA, t-SNE and UMAP Side by Side

How do we judge a map when the “right answer” has no labels? Two measures: **trustworthiness** (1.0 when every pair that looks close on the map was also close in the original space) and the accuracy of a nearest-neighbour classifier working from the 2-D coordinates only.

In [ ]:
import time, numpy as np, umap
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)                                    # 1,797 handwritten digits, 64 pixel features, 10 classes
X = StandardScaler().fit_transform(X)
print(f"{X.shape[0]} images x {X.shape[1]} features\n")

def summarise(name, emb, seconds):
    trust = trustworthiness(X, emb, n_neighbors=10)                    # 1.0 = every close pair in 2-D was also close in 64-D
    knn = cross_val_score(KNeighborsClassifier(10), emb, y, cv=5).mean()   # can 2-D neighbours still recover the digit?
    print(f"{name:28s}  trustworthiness {trust:.3f}   10-NN accuracy in 2-D {knn:.3f}   time {seconds:5.1f}s")

t0 = time.time(); pca2 = PCA(2, random_state=0).fit_transform(X); summarise("PCA (2 components)", pca2, time.time() - t0)
t0 = time.time(); ts = TSNE(2, perplexity=30, init="pca", random_state=0).fit_transform(X); summarise("t-SNE (perplexity 30)", ts, time.time() - t0)
t0 = time.time(); um = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(X); summarise("UMAP (15 neighbours)", um, time.time() - t0)
print("\nfor reference: 10-NN accuracy using all 64 original features:", round(cross_val_score(KNeighborsClassifier(10), X, y, cv=5).mean(), 3))

| Method | Trustworthiness | 10-NN accuracy in 2-D | Time | Notes |
|---|---|---|---|---|
| PCA (2 components) | 0.816 | 0.520 | 0.0 s | Linear, deterministic, instant; loses the curved structure |
| t-SNE (perplexity 30) | **0.986** | 0.952 | 3.1 s | Best local fidelity; no transform for new points |
| UMAP (15 neighbours) | 0.976 | **0.959** | 14.1 s* | Faster on large data; can place new points |

*UMAP’s time here includes one-off compilation of its numerical code; on data much larger than 1,797 rows it is generally the faster of the two, while t-SNE is faster on small data. Both nonlinear methods keep almost all neighbourhoods (trustworthiness above 0.97) that PCA loses (0.816).

## The One Knob That Matters: Perplexity

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE, trustworthiness
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True); X = StandardScaler().fit_transform(X)
print("perplexity   KL divergence   trustworthiness   10-NN acc in 2-D   number of dense groups*")
for perp in (2, 5, 30, 100, 300):
    t = TSNE(2, perplexity=perp, init="pca", random_state=0); emb = t.fit_transform(X)
    knn = cross_val_score(KNeighborsClassifier(10), emb, y, cv=5).mean()
    # crude group count: cells of a 30x30 grid that hold at least 1% of the points, merged by connectivity
    grid, _, _ = np.histogram2d(emb[:, 0], emb[:, 1], bins=30); dense = grid >= 0.01 * len(X) / 4
    from scipy.ndimage import label
    groups = label(dense, structure=np.ones((3, 3)))[1]
    print(f"{perp:8d}        {t.kl_divergence_:6.3f}          {trustworthiness(X, emb, n_neighbors=10):.3f}            {knn:.3f}              {groups}")
print("* connected dense regions on a 30x30 grid: a rough count of the 'islands' a reader would see (there are really 10 digit classes)")

Very low perplexity makes each point attend to only its 2–5 nearest neighbours, so the map shatters real clusters into dozens of little islands (62 dense regions at perplexity 2, against 10 real digit classes). Around 30–100 the map settles (14–18 regions, trustworthiness 0.984–0.986). Very high values start to blur the local structure again (accuracy back to 0.928 at 300). A standard recommendation is 5–50 for a few thousand points, but the point of the table is that you should *look at several values* and trust only what survives the change. Note that the KL divergence (0.925 down to 0.529) is not comparable across perplexities, so it cannot be used to choose one.

## Five Ways to Misread a t-SNE Map

In [ ]:
import numpy as np
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

rng = np.random.RandomState(0)
# three clusters in 20-D: A is tight and small, B is very spread out and large, C is far from both. True centre distances: A-B = 6, A-C = 40.
def cluster(centre, sd, n): return centre + rng.normal(0, sd, (n, 20))
cA, cB, cC = np.zeros(20), np.zeros(20), np.zeros(20)
cB[0] = 6; cC[1] = 40
A, B, C = cluster(cA, 0.5, 100), cluster(cB, 3.0, 800), cluster(cC, 1.0, 100)
X = np.vstack([A, B, C]); lab = np.repeat([0, 1, 2], [100, 800, 100])

emb = TSNE(2, perplexity=30, init="pca", random_state=0).fit_transform(X)
def spread(points): return np.sqrt(((points - points.mean(0)) ** 2).sum(1).mean())      # typical distance from the cluster's centre
def gap(i, j): return np.linalg.norm(emb[lab == i].mean(0) - emb[lab == j].mean(0))
print("                       true (20-D)        t-SNE map (2-D)")
print(f"spread of A (tight)      {spread(A):6.2f}            {spread(emb[lab == 0]):6.2f}")
print(f"spread of B (loose)      {spread(B):6.2f}            {spread(emb[lab == 1]):6.2f}")
print(f"ratio B / A spread       {spread(B) / spread(A):6.2f}            {spread(emb[lab == 1]) / spread(emb[lab == 0]):6.2f}")
print(f"distance A-B             {6.0:6.2f}            {gap(0, 1):6.2f}")
print(f"distance A-C             {40.0:6.2f}            {gap(0, 2):6.2f}")
print(f"ratio (A-C) / (A-B)      {40 / 6:6.2f}            {gap(0, 2) / gap(0, 1):6.2f}")

print("\nStructure from nothing: 500 points of pure Gaussian noise in 20-D (there are NO clusters)")
noise = rng.normal(0, 1, (500, 20))
for perp in (5, 50):
    e = TSNE(2, perplexity=perp, init="pca", random_state=0).fit_transform(noise)
    best = max(silhouette_score(e, KMeans(k, n_init=5, random_state=0).fit_predict(e)) for k in (3, 4, 5, 6))
    print(f"  perplexity {perp:3d}: best k-means silhouette on the 2-D map = {best:.2f}   (above 0.4 usually reads as 'real clusters')")
print(f"  the same test in the original 20-D space: {max(silhouette_score(noise, KMeans(k, n_init=5, random_state=0).fit_predict(noise)) for k in (3, 4, 5, 6)):.2f}")

1. **Cluster sizes are not real sizes.** t-SNE adapts to local density, so a loose cluster and a tight one are drawn at more similar sizes than they truly are (true spreads 2.18 versus 13.30 — a ratio of 6.11; on the map 4.91).
2. **Distances between clusters are unreliable.** The clusters A and B are truly 6 apart and A and C are 40 apart, a ratio of 6.67; the map says 12.09 and 48.14, a ratio of only 3.98. Which islands are “close” on a map tells you little.
3. **Structure appears from nothing.** On 500 points of pure Gaussian noise in 20-D, k-means finds a silhouette of 0.36 on the 2-D map at both perplexities, against 0.04 in the original space. A visible cluster is a hypothesis to test, not evidence.
4. **The axes mean nothing,** and rotating or flipping the map changes nothing.
5. **Different runs give different pictures** (next section).

## Stability and New Data

In [ ]:
import numpy as np, umap
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True); X = StandardScaler().fit_transform(X)

def neighbour_sets(emb, k=10):
    idx = NearestNeighbors(n_neighbors=k + 1).fit(emb).kneighbors(emb, return_distance=False)[:, 1:]
    return [set(r) for r in idx]

def agreement(e1, e2):
    """Average overlap between each point's 10 nearest neighbours in two different maps of the SAME data (1.0 = identical neighbourhoods)."""
    a, b = neighbour_sets(e1), neighbour_sets(e2)
    return float(np.mean([len(x & y_) / len(x | y_) for x, y_ in zip(a, b)]))

def repeat(make, seeds=(0, 1, 2, 3)):
    maps = [make(s) for s in seeds]
    pairs = [agreement(maps[i], maps[j]) for i in range(len(maps)) for j in range(i + 1, len(maps))]
    return np.mean(pairs)

print("How similar are the neighbourhoods across 4 runs that differ only in the random seed?  (Jaccard overlap of 10-NN sets)")
print(f"  t-SNE, random initialisation : {repeat(lambda s: TSNE(2, perplexity=30, init='random', random_state=s).fit_transform(X)):.3f}")
print(f"  t-SNE, PCA initialisation    : {repeat(lambda s: TSNE(2, perplexity=30, init='pca', random_state=s).fit_transform(X)):.3f}")
print(f"  UMAP                         : {repeat(lambda s: umap.UMAP(n_neighbors=15, random_state=s).fit_transform(X)):.3f}")

print("\nNew data: can the map place points it has never seen?")
rng = np.random.RandomState(0); idx = rng.permutation(len(X)); tr, te = idx[:1400], idx[1400:]
mapper = umap.UMAP(n_neighbors=15, random_state=0).fit(X[tr])
new = mapper.transform(X[te])
from sklearn.neighbors import KNeighborsClassifier
acc = KNeighborsClassifier(10).fit(mapper.embedding_, y[tr]).score(new, y[te])
print(f"  UMAP.transform on {len(te)} unseen digits, classified by 10-NN in the map: {acc:.3f} accuracy")
print("  scikit-learn's TSNE has no transform(): a new point means re-running the whole embedding")

Across four random seeds, t-SNE with random initialisation kept only 0.741 of each point’s neighbours (Jaccard overlap of 10-NN sets), and UMAP 0.566. With `init="pca"` scikit-learn’s t-SNE gave an overlap of exactly **1.000**: the starting layout is deterministic, so the seed no longer matters — recommended, and it also usually preserves more global structure. For UMAP, fix `random_state` for reproducibility (at the cost of parallel speed) and compare a few seeds before drawing conclusions. Finally, UMAP’s `transform()` placed 397 unseen digits so that a 10-NN classifier in the map got **96.7%** right; scikit-learn’s t-SNE has no `transform()`, so a new point means re-embedding everything. That makes t-SNE a *visualisation* tool and UMAP usable as a lightweight feature extractor (fit on training data only).

> **⚠ A safe workflow**
>
> (1) Standardise the features and optionally reduce to about 50 dimensions with PCA first. (2) Run t-SNE with `init="pca"` at two or three perplexities. (3) Check trustworthiness and a nearest-neighbour score against labels if you have them. (4) Treat any cluster you see as a hypothesis, and confirm it by clustering in the original space or with a supervised model. (5) Never quote cluster sizes or between-cluster distances from the picture. (6) If you feed a map into a downstream model, fit UMAP on training data only.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Setup for the exercises

The digits, standardised, are used in Exercises 1 and 2.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
X, y = load_digits(return_X_y=True); X = StandardScaler().fit_transform(X)

### Exercise 1 · Easy · How much does PCA keep?

Project the standardised digits onto their first 2 principal components and compute the **trustworthiness** of that projection with `n_neighbors=10`.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import trustworthiness
trust_pca = None   # TODO


In [ ]:
try:
    check("a number", isinstance(trust_pca, float))
    check("close to 0.816", abs(trust_pca - 0.816) < 0.02)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import trustworthiness
from sklearn.preprocessing import StandardScaler
X, y = load_digits(return_X_y=True); X = StandardScaler().fit_transform(X)
proj = PCA(2, random_state=0).fit_transform(X)
trust_pca = float(trustworthiness(X, proj, n_neighbors=10))
```

</details>


### Exercise 2 · Medium · Choose a perplexity

Run t-SNE (`init="pca"`, `random_state=0`) on the digits for perplexity 5, 30 and 100. Store the trustworthiness of each in `scores` (a dict `{perplexity: value}`) and the winner in `best`.

In [ ]:
from sklearn.manifold import TSNE, trustworthiness
scores, best = {}, None   # TODO: fill scores for 5, 30, 100 and set best to the perplexity with the highest trustworthiness


In [ ]:
try:
    check("three perplexities", set(scores) == {5, 30, 100})
    check("every map keeps neighbourhoods (> 0.95)", all(v > 0.95 for v in scores.values()))
    check("best is the argmax", best == max(scores, key=scores.get))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.manifold import TSNE, trustworthiness
scores = {}
for p in (5, 30, 100):
    emb = TSNE(2, perplexity=p, init="pca", random_state=0).fit_transform(X)
    scores[p] = float(trustworthiness(X, emb, n_neighbors=10))
best = max(scores, key=scores.get)
print(scores, best)
```

</details>


### Exercise 3 · Stretch · Structure from nothing

Generate 500 points of pure Gaussian noise in 20 dimensions (`RandomState(0)`). Compute the best k-means silhouette (k = 3 to 6) **in the original 20-D space** (`raw_sil`) and **on a 2-D t-SNE map** with perplexity 30 (`map_sil`). The map should look far more clustered than the data really is.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
noise = np.random.RandomState(0).normal(0, 1, (500, 20))
raw_sil = map_sil = None   # TODO


In [ ]:
try:
    check("raw data has no clusters (< 0.1)", raw_sil < 0.1)
    check("the map invents structure (> raw + 0.15)", map_sil > raw_sil + 0.15)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
noise = np.random.RandomState(0).normal(0, 1, (500, 20))
best_sil = lambda Z: max(silhouette_score(Z, KMeans(k, n_init=5, random_state=0).fit_predict(Z)) for k in (3, 4, 5, 6))
raw_sil = best_sil(noise)
map_sil = best_sil(TSNE(2, perplexity=30, init="pca", random_state=0).fit_transform(noise))
```

</details>


---
*Back to the course: **Machine Learning End To End → t-SNE & UMAP — Looking at High-Dimensional Data**.*